# NeuroGolf Solver Family: Fill / Additive Marking - nonlocal_multicolor


In [1]:
# Inline helper functions from submission_nbs/neurogolf_nb_common.py
"""Shared helpers for NeuroGolf submission notebooks.

The notebooks in this folder are solver-family workbooks. They should be
copied into Kaggle or run locally with the competition files available.
"""

import json
import math
import os
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np

try:
    import onnx
    import onnxruntime as ort
    from onnx import TensorProto, helper, numpy_helper
except Exception:  # Notebook analysis cells can still run without ONNX.
    onnx = None
    ort = None
    TensorProto = None
    helper = None
    numpy_helper = None




"""Shared helpers for NeuroGolf submission notebooks.

The notebooks in this folder are solver-family workbooks. They should be
copied into Kaggle or run locally with the competition files available.
"""


BATCH, CH, H, W = 1, 10, 30, 30


def default_paths():
    kaggle_dir = Path("/kaggle/input/competitions/neurogolf-2026")
    if kaggle_dir.exists():
        data_dir = kaggle_dir
        root = Path("/kaggle/working")
    else:
        root = Path.cwd()
        data_dir = root / "competition_material" / "taskfiles"
        if not data_dir.exists():
            data_dir = root / "competition_material"
    out_dir = root / "working_submission"
    out_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, out_dir


def load_task_type_map(path="/kaggle/input/datasets/prince22466/task-type-map-csv/task_type_map.csv"):
    import pandas as pd

    candidates = [Path(path), Path("task_groups/task_type_map.csv")]
    for candidate in candidates:
        if candidate.exists():
            return pd.read_csv(candidate, dtype={"task_id": str})
    raise FileNotFoundError(f"task_type_map.csv not found in: {candidates}")


def load_task_groups(path="task_groups/task_type_groups.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def family_task_ids(family, groups_path="/kaggle/input/datasets/prince22466/task-type-groups-json/task_type_groups.json"):
    groups_candidates = [Path(groups_path), Path("task_groups/task_type_groups.json")]
    for candidate in groups_candidates:
        if candidate.exists():
            groups = load_task_groups(candidate)
            return groups.get(family, [])
    raise FileNotFoundError(f"task_type_groups.json not found in: {groups_candidates}")


def task_num(task_id):
    return int(str(task_id).replace("task", ""))


def task_path(data_dir, task_id):
    data_dir = Path(data_dir)
    name = f"{task_id}.json" if str(task_id).startswith("task") else f"task{int(task_id):03d}.json"
    direct = data_dir / name
    if direct.exists():
        return direct
    nested = data_dir / "taskfiles" / name
    if nested.exists():
        return nested
    raise FileNotFoundError(name)


def load_task(data_dir, task_id):
    with task_path(data_dir, task_id).open("r", encoding="utf-8") as f:
        return json.load(f)


def all_examples(task):
    return task.get("train", []) + task.get("test", []) + task.get("arc-gen", [])


def grid_shape(grid):
    return len(grid), len(grid[0]) if grid else 0


def grid_to_tensor(grid):
    arr = np.zeros((BATCH, CH, H, W), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if 0 <= r < H and 0 <= c < W:
                arr[0, int(color), r, c] = 1.0
    return arr


def tensor_to_grid(arr):
    arr = np.asarray(arr)
    if arr.ndim == 4:
        arr = arr[0]
    grid = []
    for r in range(H):
        row = []
        for c in range(W):
            vals = np.where(arr[:, r, c] > 0.5)[0]
            row.append(int(vals[0]) if len(vals) == 1 else 0)
        while row and row[-1] == 0:
            row.pop()
        grid.append(row)
    while grid and not grid[-1]:
        grid.pop()
    return grid


def require_onnx():
    if onnx is None or helper is None or TensorProto is None or numpy_helper is None:
        raise ImportError("onnx is required to build models")


def make_initializer(name, array):
    arr = np.asarray(array, dtype=np.float16)
    return numpy_helper.from_array(arr, name=name)


def make_model(nodes, initializers, opset=10):
    require_onnx()
    # Keep competition-facing graph I/O as float32. Cast internally to float16.
    inp = helper.make_tensor_value_info("input", TensorProto.FLOAT, [BATCH, CH, H, W])
    out = helper.make_tensor_value_info("output", TensorProto.FLOAT, [BATCH, CH, H, W])
    for node in nodes:
        for i, value in enumerate(node.input):
            if value == "input":
                node.input[i] = "input_f16"
        for i, value in enumerate(node.output):
            if value == "output":
                node.output[i] = "output_f16"
    cast_in = helper.make_node("Cast", ["input"], ["input_f16"], to=TensorProto.FLOAT16)
    cast_out = helper.make_node("Cast", ["output_f16"], ["output"], to=TensorProto.FLOAT)
    graph = helper.make_graph([cast_in] + list(nodes) + [cast_out], "graph", [inp], [out], initializers)
    return helper.make_model(graph, ir_version=10, opset_imports=[helper.make_opsetid("", opset)])


def make_identity_model():
    return make_1x1_color_model({c: c for c in range(CH)})


def make_1x1_color_model(mapping):
    require_onnx()
    dt = TensorProto.FLOAT16
    weights = np.zeros((CH, CH, 1, 1), dtype=np.float16)
    bias = np.full((CH,), -0.5, dtype=np.float16)
    for ic in range(CH):
        oc = int(mapping.get(ic, ic))
        weights[oc, ic, 0, 0] = 1.0
    w = make_initializer("W", weights)
    b = make_initializer("B", bias)
    node = helper.make_node("Conv", ["input", "W", "B"], ["output"], kernel_shape=[1, 1])
    return make_model([node], [w, b])


def infer_global_color_mapping(examples):
    mapping = {}
    for ex in examples:
        inp, out = ex["input"], ex["output"]
        if grid_shape(inp) != grid_shape(out):
            return None
        for r, row in enumerate(inp):
            for c, ic in enumerate(row):
                oc = out[r][c]
                prev = mapping.get(int(ic))
                if prev is None:
                    mapping[int(ic)] = int(oc)
                elif prev != int(oc):
                    return None
    for c in range(CH):
        mapping.setdefault(c, c)
    return mapping


def train_color_remap_model(task):
    mapping = infer_global_color_mapping(all_examples(task))
    if mapping is None:
        return None, {"ok": False, "reason": "no consistent global color mapping"}
    return make_1x1_color_model(mapping), {"ok": True, "mapping": mapping}


def fixed_transform(grid, transform):
    arr = np.array(grid, dtype=int)
    if transform == "rot90":
        return np.rot90(arr, -1).tolist()
    if transform == "rot180":
        return np.rot90(arr, 2).tolist()
    if transform == "rot270":
        return np.rot90(arr, 1).tolist()
    if transform == "flip_h":
        return np.fliplr(arr).tolist()
    if transform == "flip_v":
        return np.flipud(arr).tolist()
    if transform == "transpose":
        return arr.T.tolist()
    raise ValueError(transform)


def infer_fixed_geometric_transform(examples):
    names = ["rot90", "rot180", "rot270", "flip_h", "flip_v", "transpose"]
    matches = []
    for name in names:
        if all(fixed_transform(ex["input"], name) == ex["output"] for ex in examples):
            matches.append(name)
    return matches


def save_model(model, out_dir, task_id):
    require_onnx()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{task_id}.onnx"
    onnx.save(model, path)
    return path


def run_model(model_or_path, input_grid):
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required to run validation")
    if isinstance(model_or_path, (str, Path)):
        session = ort.InferenceSession(str(model_or_path), providers=["CPUExecutionProvider"])
    else:
        session = ort.InferenceSession(model_or_path.SerializeToString(), providers=["CPUExecutionProvider"])
    output = session.run(["output"], {"input": grid_to_tensor(input_grid)})[0]
    return (output > 0).astype(np.float32)


def visible_validation_summary(model_or_path, task, max_examples=None):
    examples = all_examples(task)
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong_expected = None
    first_wrong_actual =None
    print("len of examples: ", len(examples))

    count_ex =1
    for ex in examples:
        if count_ex % 10 ==0:
            print(f"example {count_ex}")
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong_expected is None:
                first_wrong_expected = ex
                first_wrong_actual = actual

        count_ex = count_ex +1
        
    return {"right": right, "wrong": wrong, 
            "first_wrong_expected": first_wrong_expected, "first_wrong_actual":first_wrong_actual}


def split_examples(task):
    return {
        "train": task.get("train", []),
        "test": task.get("test", []),
        "arc_gen": task.get("arc-gen", []),
    }


def validation_summary_for_examples(model_or_path, examples, max_examples=None):
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    total = right + wrong
    accuracy = right / total if total else None
    return {"right": right, "wrong": wrong, "total": total, "accuracy": accuracy, "first_wrong": first_wrong}


def split_validation_summary(model_or_path, task, max_examples=None):
    rows = {}
    for split, examples in split_examples(task).items():
        summary = validation_summary_for_examples(model_or_path, examples, max_examples=max_examples)
        summary.pop("first_wrong", None)
        rows[split] = summary
    visible = validation_summary_for_examples(model_or_path, all_examples(task), max_examples=max_examples)
    visible.pop("first_wrong", None)
    rows["visible_all"] = visible
    return rows


def count_model_params(model_or_path):
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    params = 0
    for init in model.graph.initializer:
        if init.dims:
            params += math.prod(init.dims)
        else:
            params += 1
    for node in model.graph.node:
        if node.op_type != "Constant":
            continue
        for attr in node.attribute:
            if attr.name == "value":
                params += math.prod(attr.t.dims) if attr.t.dims else 1
            elif attr.name == "value_floats":
                params += len(attr.floats)
            elif attr.name == "value_ints":
                params += len(attr.ints)
            elif attr.name == "value_strings":
                params += len(attr.strings)
    return int(params)


def model_architecture_summary(model_or_path):
    require_onnx()
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    op_counts = Counter(node.op_type for node in model.graph.node)
    init_shapes = {init.name: list(init.dims) for init in model.graph.initializer}
    return {
        "ir_version": model.ir_version,
        "opsets": {op.domain or "ai.onnx": op.version for op in model.opset_import},
        "nodes": len(model.graph.node),
        "op_counts": dict(op_counts),
        "initializers": init_shapes,
        "params": count_model_params(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
    }


def approximate_memory_from_model_shapes(model_or_path):
    """Approximate scored tensor memory from static value_info shapes.

    The official helper uses ONNX Runtime profiling to refine tensor memory.
    This approximation is useful in notebooks before running the full profiler.
    """
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    inferred = onnx.shape_inference.infer_shapes(model)
    graph = inferred.graph
    total = 0
    for value in list(graph.value_info):
        tensor_type = value.type.tensor_type
        if not tensor_type.HasField("shape"):
            continue
        dims = []
        for dim in tensor_type.shape.dim:
            if not dim.HasField("dim_value") or dim.dim_value <= 0:
                dims = []
                break
            dims.append(dim.dim_value)
        if dims:
            total += math.prod(dims) * 2
    return int(total)


def runtime_memory_profile(model_or_path, sample_input_grid):
    """Return an approximate competition memory/param profile.

    This uses ONNX Runtime profiling if available. It is not a replacement for
    the official Kaggle validator, but it tracks the same concerns: parameters,
    intermediate tensor memory, and file size.
    """
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required for runtime memory profiling")
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    options = ort.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    session = ort.InferenceSession(model.SerializeToString(), options, providers=["CPUExecutionProvider"])
    session.run(["output"], {"input": grid_to_tensor(sample_input_grid)})
    trace_path = session.end_profiling()
    trace_memory = 0
    try:
        with open(trace_path, "r", encoding="utf-8") as f:
            trace = json.load(f)
        for event in trace:
            args = event.get("args", {})
            for shape_dict in args.get("output_type_shape", []) or []:
                for dims in shape_dict.values():
                    if dims and all(isinstance(d, int) and d > 0 for d in dims):
                        trace_memory += math.prod(dims) * 2
    except Exception:
        trace_memory = approximate_memory_from_model_shapes(model)
    return {
        "params": count_model_params(model),
        "runtime_memory_bytes": int(trace_memory),
        "static_memory_bytes": approximate_memory_from_model_shapes(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
        "profile_trace_path": trace_path,
    }


def model_report(model_or_path, task=None, sample_input_grid=None, max_validation_examples=None):
    report = {"architecture": model_architecture_summary(model_or_path)}
    if task is not None:
        report["performance"] = split_validation_summary(
            model_or_path,
            task,
            max_examples=max_validation_examples,
        )
        if sample_input_grid is None:
            examples = all_examples(task)
            if examples:
                sample_input_grid = examples[0]["input"]
    if sample_input_grid is not None and ort is not None:
        report["memory_profile"] = runtime_memory_profile(model_or_path, sample_input_grid)
    else:
        report["memory_profile"] = {
            "params": report["architecture"]["params"],
            "static_memory_bytes": approximate_memory_from_model_shapes(model_or_path),
            "runtime_memory_bytes": None,
            "file_size_bytes": report["architecture"]["file_size_bytes"],
            "profile_trace_path": None,
        }
    return report


def create_submission_zip(model_dir, zip_path=None):
    model_dir = Path(model_dir)
    if zip_path is None:
        zip_path = model_dir / "submission.zip"
    else:
        zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(model_dir.glob("task*.onnx")):
            zf.write(path, path.name)
    return zip_path


def build_family_submission(family, trainer, data_dir, out_dir, fallback_identity=False, validate=False, task_ids_override=None):
    task_ids = list(task_ids_override) if task_ids_override is not None else family_task_ids(family)
    rows = []
    for task_id in task_ids:
        task = load_task(data_dir, task_id)
        model, info = trainer(task)
        if model is None and fallback_identity:
            model = make_identity_model()
            info = {**info, "fallback": "identity"}
        if model is None:
            rows.append({"task_id": task_id, "saved": False, **info})
            continue
        path = save_model(model, out_dir, task_id)
        row = {"task_id": task_id, "saved": True, "path": str(path), **info}
        if validate:
            try:
                row.update({f"visible_{k}": v for k, v in visible_validation_summary(path, task).items() if k != "first_wrong"})
            except Exception as exc:
                row["visible_error"] = repr(exc)
        rows.append(row)
    zip_path = create_submission_zip(out_dir)
    return rows, zip_path


In [2]:
# ONNX dependency setup for model export.
# Dry-run rule fitting can run without ONNX, but task145 export needs ONNX Runtime and scikit-learn.
import importlib.util
import subprocess
import sys

required_packages = {
    'onnx': 'onnx',
    'onnxruntime': 'onnxruntime',
    'sklearn': 'scikit-learn',
}
missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing ONNX/export packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import onnx
import onnxruntime as ort
from onnx import TensorProto, helper, numpy_helper

print('onnx:', onnx.__version__)
print('onnxruntime:', ort.__version__)

Installing missing ONNX/export packages: ['onnxruntime']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 63.8 MB/s eta 0:00:00
onnx: 1.20.1
onnxruntime: 1.27.0


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from sklearn.tree import DecisionTreeClassifier
    SKLEARN_IMPORT_ERROR = None
except Exception as exc:
    DecisionTreeClassifier = None
    SKLEARN_IMPORT_ERROR = repr(exc)


def require_tree_export_dependencies():
    if DecisionTreeClassifier is None:
        raise ImportError(f'scikit-learn import failed: {SKLEARN_IMPORT_ERROR}')

H=W=30; CH=10
FORBIDDEN={'Loop','Scan','NonZero','Unique','Script','Function'}
torch.set_num_threads(2)

def grid_to_tensor(grid):
    arr=np.zeros((1,CH,H,W),np.float32)
    for r,row in enumerate(grid[:H]):
        for c,v in enumerate(row[:W]):
            arr[0,int(v),r,c]=1.0
    return arr

def expected_tensor(grid):
    return grid_to_tensor(grid)

def all_examples(task):
    return task.get('train',[])+task.get('test',[])+task.get('arc-gen',[])

class BaseRule(nn.Module):
    def __init__(self):
        super().__init__()
        rr=torch.arange(H,dtype=torch.float32).view(1,1,H,1).expand(1,1,H,W)
        cc=torch.arange(W,dtype=torch.float32).view(1,1,1,W).expand(1,1,H,W)
        self.register_buffer('rr',rr)
        self.register_buffer('cc',cc)
        k4=torch.zeros((1,1,3,3),dtype=torch.float32); k4[0,0,1,1]=1; k4[0,0,0,1]=1; k4[0,0,2,1]=1; k4[0,0,1,0]=1; k4[0,0,1,2]=1
        self.register_buffer('k4',k4)
        border=torch.zeros((1,1,H,W),dtype=torch.float32); border[:,:,0,:]=1; border[:,:,-1,:]=1; border[:,:,:,0]=1; border[:,:,:,-1]=1
        self.register_buffer('border',border)
    def active(self,x): return (x.sum(1,keepdim=True)>0).float()
    def mask(self,x,k): return x[:,k:k+1]
    def nonzero(self,x): return x[:,1:,:,:].sum(1,keepdim=True).clamp(0,1)
    def dilate4(self,m): return (F.conv2d(m,self.k4,padding=1)>0).float()
    def flood(self, allowed, seed, steps=64):
        r=seed*allowed
        for _ in range(steps):
            r=self.dilate4(r)*allowed
        return r
    def exterior_zero(self,zero,active):
        inactive=(1-active)
        seed=zero*((self.dilate4(inactive)+self.border)>0).float()
        return self.flood(zero,seed,64)
    def onehot_from_masks(self,masks,active=None):
        # masks dict color -> [B,1,H,W]
        outs=[]
        for k in range(CH):
            m=masks.get(k)
            if m is None:
                m=torch.zeros_like(next(iter(masks.values())))
            outs.append(m)
        y=torch.cat(outs,1)
        if active is not None: y=y*active
        return y

class Task187(BaseRule):
    def forward(self,x):
        active=self.active(x); zero=self.mask(x,0); nz=self.nonzero(x)
        exterior=self.exterior_zero(zero,active)
        enclosed=zero*(1-exterior)
        masks={}
        for k in range(1,CH): masks[k]=self.mask(x,k)
        masks[2]=masks.get(2,0)+enclosed
        masks[3]=masks.get(3,0)+exterior
        return self.onehot_from_masks(masks,active)

class Task198(BaseRule):
    def forward(self,x):
        active=self.active(x); zero=self.mask(x,0); nz=self.nonzero(x)
        row_nz=nz.sum(3,keepdim=True) # [B,1,H,1]
        row_active=active.sum(3,keepdim=True)
        line_row=(row_nz*2 > row_active).float()
        col_nz=nz.sum(2,keepdim=True) # [B,1,1,W]
        col_active=active.sum(2,keepdim=True)
        line_col=(col_nz*2 > col_active).float()
        gap_seed=zero*((line_row+line_col)>0).float()
        color4=self.flood(zero,gap_seed,64)
        color3=zero*(1-color4)
        masks={}
        for k in range(1,CH): masks[k]=self.mask(x,k)
        masks[3]=masks.get(3,0)+color3
        masks[4]=masks.get(4,0)+color4
        return self.onehot_from_masks(masks,active)

class Task369(BaseRule):
    def forward(self,x):
        active=self.active(x); zero=self.mask(x,0)
        deg=(F.conv2d(zero,self.k4,padding=1)-zero).clamp(0,4)
        iso=zero*(deg==0).float()
        deg2=zero*(deg>=2).float()
        near_deg2=self.dilate4(deg2)*zero
        size3=near_deg2
        size2=zero*(1-iso)*(1-size3)
        masks={5:self.mask(x,5), 1:size3, 2:size2, 3:iso}
        return self.onehot_from_masks(masks,active)

class Task256(BaseRule):
    def forward(self,x):
        active=self.active(x); zero=self.mask(x,0); line=self.mask(x,2)
        row_has=(line.sum(3,keepdim=True)>0).float()
        R=(row_has*self.rr[:,:,:,0:1]).sum((2,3),keepdim=True)
        L=line.sum((2,3),keepdim=True)
        above=(self.rr < R).float(); below=(self.rr > R).float()
        dist_up=(R-self.rr); dist_down=(self.rr-R)
        color3=zero*above*(self.cc < (L+dist_up)).float()
        color1=zero*below*(self.cc < (L-dist_down)).float()
        masks={2:line,1:color1,3:color3,0:zero*(1-color1)*(1-color3)}
        return self.onehot_from_masks(masks,active)

class Task226(BaseRule):
    def forward(self,x):
        active=self.active(x); zero=self.mask(x,0); line=self.mask(x,5)
        row_count=line.sum(3,keepdim=True); row_active=active.sum(3,keepdim=True)
        sep_row=(row_count*2 > row_active).float()
        col_count=line.sum(2,keepdim=True); col_active=active.sum(2,keepdim=True)
        sep_col=(col_count*2 > col_active).float()
        row_seg=torch.cumsum(sep_row,dim=2)-sep_row
        col_seg=torch.cumsum(sep_col,dim=3)-sep_col
        fillbase=zero*(1-sep_row)*(1-sep_col)
        n_sep_rows=sep_row.sum((2,3),keepdim=True)
        n_sep_cols=sep_col.sum((2,3),keepdim=True)
        mid_row=torch.floor(n_sep_rows/2)
        mid_col=torch.floor(n_sep_cols/2)
        last_row=n_sep_rows
        last_col=n_sep_cols
        c1=fillbase*(row_seg==0).float()*(col_seg==0).float()
        c2=fillbase*(row_seg==mid_row).float()*(col_seg==mid_col).float()
        c3=fillbase*(row_seg==last_row).float()*(col_seg==last_col).float()
        masks={5:line,1:c1,2:c2,3:c3,0:zero*(1-c1)*(1-c2)*(1-c3)}
        return self.onehot_from_masks(masks,active)

class Task055(BaseRule):
    def forward(self,x):
        active=self.active(x); zero=self.mask(x,0); line=self.mask(x,8)
        row_count=line.sum(3,keepdim=True); row_active=active.sum(3,keepdim=True)
        sep_row=(row_count*2 > row_active).float()
        col_count=line.sum(2,keepdim=True); col_active=active.sum(2,keepdim=True)
        sep_col=(col_count*2 > col_active).float()
        row_seg=torch.cumsum(sep_row,dim=2)-sep_row
        col_seg=torch.cumsum(sep_col,dim=3)-sep_col
        fill=zero*(1-sep_row)*(1-sep_col)
        c2=fill*(row_seg==0).float()*(col_seg==1).float()
        c4=fill*(row_seg==1).float()*(col_seg==0).float()
        c6=fill*(row_seg==1).float()*(col_seg==1).float()
        c3=fill*(row_seg==1).float()*(col_seg==2).float()
        c1=fill*(row_seg==2).float()*(col_seg==1).float()
        union=(c1+c2+c3+c4+c6).clamp(0,1)
        masks={8:line,1:c1,2:c2,3:c3,4:c4,6:c6,0:zero*(1-union)}
        return self.onehot_from_masks(masks,active)

class Task302(BaseRule):
    def coverage(self,enclosed,k):
        ker=torch.ones((1,1,k,k),dtype=enclosed.dtype,device=enclosed.device)
        full=(F.conv2d(enclosed,ker)==k*k).float()
        cov=(F.conv_transpose2d(full,ker)>0).float()
        return cov
    def forward(self,x):
        active=self.active(x); zero=self.mask(x,0); line=self.mask(x,5)
        exterior=self.exterior_zero(zero,active); enclosed=zero*(1-exterior)
        cov3=self.coverage(enclosed,3); cov2=self.coverage(enclosed,2)
        c8=enclosed*cov3
        c7=enclosed*(1-c8)*cov2
        c6=enclosed*(1-c8)*(1-c7)
        masks={5:line,6:c6,7:c7,8:c8,0:zero*(1-enclosed)}
        return self.onehot_from_masks(masks,active)

class Task204(BaseRule):
    def coverage(self,enclosed,k):
        ker=torch.ones((1,1,k,k),dtype=enclosed.dtype,device=enclosed.device)
        full=(F.conv2d(enclosed,ker)==k*k).float()
        return (F.conv_transpose2d(full,ker)>0).float()
    def forward(self,x):
        active=self.active(x); zero=self.mask(x,0); line=self.mask(x,1)
        exterior=self.exterior_zero(zero,active); enclosed=zero*(1-exterior)
        assigned=torch.zeros_like(enclosed); c2=torch.zeros_like(enclosed); c7=torch.zeros_like(enclosed)
        for k in [8,7,6,5,4,3,2,1]:
            cov=self.coverage(enclosed,k)*(1-assigned)
            if k%2==0: c2=c2+cov
            else: c7=c7+cov
            assigned=assigned+cov
        masks={1:line,2:c2,7:c7,0:zero*(1-enclosed)}
        return self.onehot_from_masks(masks,active)

class Task349(BaseRule):
    def shift(self,m,dr,dc):
        z=torch.zeros_like(m)
        r0=max(0,-dr); r1=min(H,H-dr); c0=max(0,-dc); c1=min(W,W-dc)
        z[:,:,r0+dr:r1+dr,c0+dc:c1+dc]=m[:,:,r0:r1,c0:c1]
        return z
    def exact_topleft(self,nine,s):
        tl=torch.ones_like(nine)
        for dr in range(s):
            for dc in range(s):
                tl=tl*self.shift(nine,-dr,-dc)  # cell (r+dr,c+dc) is nine -> shift up/left to tl
        # no 9 immediately above/left; no 9 just after square down/right at top-left axes
        no_above=1-self.shift(nine,1,0)  # nine at r-1,c shifted down to r,c
        no_left=1-self.shift(nine,0,1)
        no_down=1-self.shift(nine,-s,0)  # nine at r+s,c shifted up
        no_right=1-self.shift(nine,0,-s)
        return tl*no_above*no_left*no_down*no_right
    def transposed_from_tl(self,tl,kernel,margin):
        m=margin
        padded=F.pad(tl,(m,m,m,m))
        out=F.conv_transpose2d(padded,kernel)
        return out[:,:,2*m:2*m+H,2*m:2*m+W]
    def forward(self,x):
        active=self.active(x); zero=self.mask(x,0); nine=self.mask(x,9)
        frame=torch.zeros_like(nine); shadow=torch.zeros_like(nine)
        for s in [2,4,6,8]:
            m=s//2
            tl=self.exact_topleft(nine,s)
            # frame kernel: expanded square minus original central square
            K=s+2*m
            fk=torch.ones((1,1,K,K),dtype=x.dtype,device=x.device)
            fk[:,:,m:m+s,m:m+s]=0
            frame=frame+self.transposed_from_tl(tl,fk,m)
            sk=torch.zeros((1,1,H+s+m,K),dtype=x.dtype,device=x.device)
            # easier: large kernel from tl: rows from s+m to H+m+s, cols m:m+s
            sk[:,:,s+m: s+m+H, m:m+s]=1
            # use crop similar but kernel large; crop top-left
            padded=F.pad(tl,(m,m,m,m))
            sh=F.conv_transpose2d(padded,sk)
            shadow=shadow+sh[:,:,2*m:2*m+H,2*m:2*m+W]
        frame=(frame>0).float()*zero
        shadow=(shadow>0).float()*zero*(1-frame)
        masks={9:nine,3:frame,1:shadow,0:zero*(1-frame)*(1-shadow)}
        return self.onehot_from_masks(masks,active)

SEMANTIC_CLASSES={
 'task055':Task055, 'task187':Task187, 'task198':Task198, 'task204':Task204,
 'task226':Task226, 'task256':Task256, 'task302':Task302, 'task349':Task349,
 'task369':Task369,
}

def export_torch_model(task_id,out_path):
    model=SEMANTIC_CLASSES[task_id]().eval()
    dummy=torch.zeros(1,CH,H,W,dtype=torch.float32)
    torch.onnx.export(model,dummy,str(out_path),input_names=['input'],output_names=['output'],opset_version=17,dynamic_axes=None,do_constant_folding=True,dynamo=False)
    # fix ir version maybe ORT supports
    onnx_model=onnx.load(str(out_path))
    onnx.checker.check_model(onnx_model)
    return out_path

# Raw-grid decision tree for task145

def grid_int(grid):
    a=np.full((H,W),-1,np.int16); h=min(len(grid),H); w=min(len(grid[0]),W)
    for r,row in enumerate(grid[:H]):
        for c,v in enumerate(row[:W]): a[r,c]=int(v)
    return a,h,w

def raw_tree_dataset(task):
    X=[]; y=[]
    rr=np.arange(H)[:,None].repeat(W,1); cc=np.arange(W)[None,:].repeat(H,0)
    for ex in all_examples(task):
        a,h,w=grid_int(ex['input']); flat=np.where(a>=0,a,0).ravel().astype(np.float32)
        out,_,_=grid_int(ex['output']); active=a>=0
        coords=np.stack([rr[active], cc[active], np.full(active.sum(),h), np.full(active.sum(),w)],1).astype(np.float32)
        X.append(np.concatenate([coords, np.repeat(flat[None,:], active.sum(), axis=0)],1)); y.append(out[active])
    return np.concatenate(X), np.concatenate(y)

def make_tree_ensemble_classifier_node(clf):
    tree = clf.tree_
    classes = [int(c) for c in clf.classes_]
    class_to_index = {label: idx for idx, label in enumerate(classes)}
    nodes_treeids = []
    nodes_nodeids = []
    nodes_featureids = []
    nodes_modes = []
    nodes_values = []
    nodes_truenodeids = []
    nodes_falsenodeids = []
    nodes_missing_value_tracks_true = []
    nodes_hitrates = []
    class_treeids = []
    class_nodeids = []
    class_ids = []
    class_weights = []

    for node_id in range(tree.node_count):
        left = int(tree.children_left[node_id])
        right = int(tree.children_right[node_id])
        is_leaf = left == right or left < 0
        nodes_treeids.append(0)
        nodes_nodeids.append(node_id)
        nodes_featureids.append(0 if is_leaf else int(tree.feature[node_id]))
        nodes_modes.append('LEAF' if is_leaf else 'BRANCH_LEQ')
        nodes_values.append(0.0 if is_leaf else float(tree.threshold[node_id]))
        nodes_truenodeids.append(0 if is_leaf else left)
        nodes_falsenodeids.append(0 if is_leaf else right)
        nodes_missing_value_tracks_true.append(0)
        nodes_hitrates.append(1.0)

        if is_leaf:
            counts = tree.value[node_id][0]
            total = float(np.sum(counts)) or 1.0
            for class_pos, label in enumerate(classes):
                class_treeids.append(0)
                class_nodeids.append(node_id)
                class_ids.append(class_to_index[label])
                class_weights.append(float(counts[class_pos]) / total)

    return helper.make_node(
        'TreeEnsembleClassifier',
        ['features'],
        ['label', 'probabilities'],
        domain='ai.onnx.ml',
        classlabels_int64s=classes,
        nodes_treeids=nodes_treeids,
        nodes_nodeids=nodes_nodeids,
        nodes_featureids=nodes_featureids,
        nodes_modes=nodes_modes,
        nodes_values=nodes_values,
        nodes_truenodeids=nodes_truenodeids,
        nodes_falsenodeids=nodes_falsenodeids,
        nodes_missing_value_tracks_true=nodes_missing_value_tracks_true,
        nodes_hitrates=nodes_hitrates,
        class_treeids=class_treeids,
        class_nodeids=class_nodeids,
        class_ids=class_ids,
        class_weights=class_weights,
        post_transform='NONE',
    )


def make_raw_tree_model(task):
    require_tree_export_dependencies()
    X,y=raw_tree_dataset(task)
    clf=DecisionTreeClassifier(random_state=42, min_samples_leaf=1)
    clf.fit(X,y)
    tree_node=make_tree_ensemble_classifier_node(clf)
    # build pre/post graph around the tree node
    inputs=[helper.make_tensor_value_info('input', TensorProto.FLOAT, [1,CH,H,W])]
    outputs=[helper.make_tensor_value_info('output', TensorProto.FLOAT, [1,CH,H,W])]
    inits=[]; nodes=[]
    def init(name,arr):
        t=numpy_helper.from_array(np.asarray(arr), name); inits.append(t); return name
    init('color_values', np.arange(CH,dtype=np.float32).reshape(1,CH,1,1))
    init('shape_1_900', np.array([1,H*W],dtype=np.int64))
    init('shape_900_900', np.array([H*W,H*W],dtype=np.int64))
    init('shape_900_1', np.array([H*W,1],dtype=np.int64))
    init('shape_1_1_30_30', np.array([1,1,H,W],dtype=np.int64))
    init('shape_1_30_30_10', np.array([1,H,W,CH],dtype=np.int64))
    init('shape_900_10', np.array([H*W,CH],dtype=np.int64))
    init('depth10', np.array(10,dtype=np.int64))
    init('onehot_values', np.array([0.0,1.0],dtype=np.float32))
    coords=np.zeros((H*W,2),np.float32)
    for r in range(H):
        for c in range(W): coords[r*W+c]=[r,c]
    init('coord_rc', coords)
    nodes += [
        helper.make_node('Mul',['input','color_values'],['weighted']),
        helper.make_node('ReduceSum',['weighted'],['grid_color'],axes=[1],keepdims=0),
        helper.make_node('Reshape',['grid_color','shape_1_900'],['flat_grid']),
        helper.make_node('Expand',['flat_grid','shape_900_900'],['flat_rep']),
        helper.make_node('ReduceSum',['input'],['active_sum'],axes=[1],keepdims=0),
        helper.make_node('Greater',['active_sum', init('zero_scalar',np.array(0,dtype=np.float32))],['active_bool']),
        helper.make_node('Cast',['active_bool'],['active_float'],to=TensorProto.FLOAT),
        helper.make_node('ReduceMax',['active_float'],['row_has'],axes=[2],keepdims=0),
        helper.make_node('ReduceMax',['active_float'],['col_has'],axes=[1],keepdims=0),
        helper.make_node('ReduceSum',['row_has'],['h_scalar'],axes=[1],keepdims=1),
        helper.make_node('ReduceSum',['col_has'],['w_scalar'],axes=[1],keepdims=1),
        helper.make_node('Expand',['h_scalar','shape_900_1'],['h_rep']),
        helper.make_node('Expand',['w_scalar','shape_900_1'],['w_rep']),
        helper.make_node('Concat',['coord_rc','h_rep','w_rep','flat_rep'],['features'],axis=1),
    ]
    # tree ensemble node, only label output is used by the post-processing graph
    nodes.append(tree_node)
    nodes += [
        helper.make_node('OneHot',['label','depth10','onehot_values'],['oh_flat'],axis=-1),
        helper.make_node('Reshape',['oh_flat','shape_1_30_30_10'],['oh_nhwc']),
        helper.make_node('Transpose',['oh_nhwc'],['oh_nchw'],perm=[0,3,1,2]),
        helper.make_node('Reshape',['active_float','shape_1_1_30_30'],['active_nchw']),
        helper.make_node('Mul',['oh_nchw','active_nchw'],['output']),
    ]
    graph=helper.make_graph(nodes,'raw_grid_tree_task145',inputs,outputs,initializer=inits)
    model=helper.make_model(graph, opset_imports=[helper.make_opsetid('',12), helper.make_opsetid('ai.onnx.ml',3)])
    model.ir_version=8
    onnx.checker.check_model(model)
    return model, {'nodes': clf.tree_.node_count, 'depth': clf.get_depth()}

def export_task145(task,out_path):
    model,info=make_raw_tree_model(task)
    onnx.save(model,str(out_path))
    return info

def run_model_session(sess,grid):
    out=sess.run(['output'],{'input':grid_to_tensor(grid)})[0]
    return (out>0).astype(np.float32)

def validate(path,task):
    sess=ort.InferenceSession(str(path),providers=['CPUExecutionProvider'])
    right=0; total=0; first=None
    for i,ex in enumerate(all_examples(task)):
        total+=1
        exp=expected_tensor(ex['output'])
        got=run_model_session(sess,ex['input'])
        if np.array_equal(exp,got): right+=1
        elif first is None: first=i
    return right,total,first

def op_counts(path):
    m=onnx.load(str(path)); c={}
    for n in m.graph.node: c[n.op_type]=c.get(n.op_type,0)+1
    return c

def build_all(task_paths,outdir):
    outdir=Path(outdir); outdir.mkdir(parents=True,exist_ok=True)
    rows=[]
    for p in task_paths:
        tid=Path(p).stem
        task=json.load(open(p))
        out=outdir/f'{tid}.onnx'
        if tid=='task145':
            info=export_task145(task,out); trainer='raw_grid_decision_tree_global_feature'
        else:
            export_torch_model(tid,out); info={}; trainer='semantic_static_onnx_rule'
        right,total,first=validate(out,task)
        ops=op_counts(out); bad=sorted(FORBIDDEN & set(ops))
        size=os.path.getsize(out)
        rows.append({'task_id':tid,'path':str(out),'right':right,'total':total,'first_wrong':first,'size':size,'forbidden':bad,'ops':ops,'trainer':trainer,**info})
        print(rows[-1], flush=True)
    zip_path=outdir/'arc_task_models.zip'
    with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(outdir.glob('task*.onnx')): zf.write(p,p.name)
    return rows,zip_path

In [4]:
from pathlib import Path
import ast
import json
import pandas as pd

ROOT = Path.cwd()

FAMILY = 'fill_enclosed_regions'
SUBTYPE = 'nonlocal_multicolor'
TASK_ID = 'task369'
MODEL_VERSION = 'fill-additive-nonlocal-multicolor-v11-task369-semantic-static-rule'
DATA_DIR, BASE_OUT_DIR = default_paths()
OUT_DIR = BASE_OUT_DIR / f'{FAMILY}_{SUBTYPE}_{TASK_ID}_v11'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR =', DATA_DIR)
print('OUT_DIR =', OUT_DIR)
print('MODEL_VERSION =', MODEL_VERSION)

DATA_DIR = /kaggle/input/competitions/neurogolf-2026
OUT_DIR = /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_multicolor_task369_v11
MODEL_VERSION = fill-additive-nonlocal-multicolor-v11-task369-semantic-static-rule


In [5]:
# task369-only selection for the nonlocal_multicolor subtype.
task_map = load_task_type_map()
family_df = task_map[task_map.primary_family == FAMILY].copy()


def parse_color_list(value):
    if pd.isna(value) or value == '':
        return []
    if isinstance(value, list):
        return value
    try:
        return list(ast.literal_eval(str(value)))
    except Exception:
        return []

family_df['parsed_new_output_colors'] = family_df['new_output_color_list'].apply(parse_color_list)
nonlocal_multicolor_df = family_df[family_df['task_id'].isin([TASK_ID])].copy().reset_index(drop=True)
task_ids = [TASK_ID]

print('family:', FAMILY)
print('subtype:', SUBTYPE)
print('family tasks:', len(family_df))
print('selected task369 tasks:', len(task_ids))
print(task_ids)
display(nonlocal_multicolor_df.head(10))

family: fill_enclosed_regions
subtype: nonlocal_multicolor
family tasks: 59
selected task369 tasks: 1
['task369']


,task_id,task_num,primary_family,confidence,candidate_flags,n_train,n_test,n_arc_gen,n_examples,shape_relation,...,mapping_conflicts,fixed_geometric_transforms,local_3x3_score,local_3x3_conflicts,local_3x3_samples,input_nonzero_preserved_ratio,added_nonzero_cells,changed_cells,notes,parsed_new_output_colors
0,task369,369,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,3,1,261,265,same_shape_variable_size,...,5785,NaN,0.9445,333,6000,1.0,7121,7121,Same shape; input is mostly preserved while ne...,"[1, 2, 3]"


In [6]:
# Inspect one scoped task quickly.
if task_ids:
    sample_task_id = task_ids[0]
    sample_task = load_task(DATA_DIR, sample_task_id)
    print(sample_task_id, 'examples:', len(all_examples(sample_task)))
    print('first input shape:', grid_shape(sample_task['train'][0]['input']))
    print('first output shape:', grid_shape(sample_task['train'][0]['output']))
    print('first input:', sample_task['train'][0]['input'])
    print('first output:', sample_task['train'][0]['output'])
else:
    print('No tasks currently mapped to this subtype.')

task369 examples: 265
first input shape: (10, 10)
first output shape: (10, 10)
first input: [[5, 5, 5, 5, 0, 5, 5, 5, 0, 5], [0, 0, 5, 5, 5, 5, 5, 5, 5, 5], [0, 5, 5, 5, 5, 5, 0, 0, 5, 0], [5, 5, 0, 5, 5, 5, 5, 0, 5, 0], [5, 5, 5, 5, 0, 0, 5, 5, 5, 5], [0, 5, 0, 5, 5, 5, 5, 0, 5, 0], [0, 5, 5, 5, 0, 0, 5, 5, 5, 0], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 0], [0, 5, 5, 5, 5, 5, 5, 0, 5, 0]]
first output: [[5, 5, 5, 5, 3, 5, 5, 5, 3, 5], [1, 1, 5, 5, 5, 5, 5, 5, 5, 5], [1, 5, 5, 5, 5, 5, 1, 1, 5, 2], [5, 5, 3, 5, 5, 5, 5, 1, 5, 2], [5, 5, 5, 5, 2, 2, 5, 5, 5, 5], [2, 5, 3, 5, 5, 5, 5, 3, 5, 2], [2, 5, 5, 5, 2, 2, 5, 5, 5, 2], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 2], [3, 5, 5, 5, 5, 5, 5, 3, 5, 2]]


In [7]:
# nonlocal_multicolor selection table: this notebook version is responsible only for task055.
selection_cols = [
    'task_id',
    'confidence',
    'n_train',
    'n_test',
    'n_arc_gen',
    'shape_relation',
    'input_shape_modes',
    'output_shape_modes',
    'input_color_list',
    'output_color_list',
    'new_output_color_list',
    'local_3x3_score',
    'local_3x3_conflicts',
    'added_nonzero_cells',
    'candidate_flags',
]
fill_selection = nonlocal_multicolor_df[selection_cols].reset_index(drop=True)
print('selected nonlocal_multicolor fill/additive tasks:', len(fill_selection))
display(fill_selection)

selected nonlocal_multicolor fill/additive tasks: 1


,task_id,confidence,n_train,n_test,n_arc_gen,shape_relation,input_shape_modes,output_shape_modes,input_color_list,output_color_list,new_output_color_list,local_3x3_score,local_3x3_conflicts,added_nonzero_cells,candidate_flags
0,task369,medium,3,1,261,same_shape_variable_size,10x10:265,10x10:265,"[0,5]","[1,2,3,5]","[1,2,3]",0.9445,333,7121,adds_new_color_preserves_input|new_output_colors


In [8]:
# Correctness/export plan for task369.
if 'task_ids' not in globals():
    task_ids = [TASK_ID]

TASK_MODEL_BUILDERS = {TASK_ID: 'export_torch_model'}
plan_rows = []
for task_id in task_ids:
    task = load_task(DATA_DIR, task_id)
    plan_rows.append({
        'task_id': task_id,
        'builder': TASK_MODEL_BUILDERS.get(task_id),
        'n_train': len(task.get('train', [])),
        'n_test': len(task.get('test', [])),
        'n_arc_gen': len(task.get('arc-gen', [])),
        'status': 'ready_to_export' if task_id in TASK_MODEL_BUILDERS else 'missing_builder',
    })

plan_df = pd.DataFrame(plan_rows)
display(plan_df)
assert set(task_ids) == {TASK_ID}, task_ids
assert all(row['status'] == 'ready_to_export' for row in plan_rows), plan_rows

,task_id,builder,n_train,n_test,n_arc_gen,status
0,task369,export_torch_model,3,1,261,ready_to_export


In [9]:
# task-specific model export wrapper
# This incorporates the task369 semantic static torch exporter used by task369_solution.ipynb.


def build_task369_model(task_id, model_path):
    if task_id != TASK_ID:
        raise ValueError(f'this notebook only builds {TASK_ID}, got {task_id}')
    export_torch_model(task_id, model_path)
    return {}


def validate_task369_model(model_path, task):
    right, total, first_wrong = validate(model_path, task)
    ops = op_counts(model_path)
    forbidden_present = sorted(FORBIDDEN & set(ops))
    size_bytes = Path(model_path).stat().st_size
    return {
        'right': right,
        'total': total,
        'accuracy': right / total if total else None,
        'first_wrong': first_wrong,
        'file_size_bytes': size_bytes,
        'under_1_4mb': size_bytes < 1_400_000,
        'forbidden_ops_present': forbidden_present,
        'op_counts': ops,
    }


EXPORTABLE_PREDICTORS = {TASK_ID: ('task369_semantic_static_torch_export', None)}
SIMULATOR_ONLY_PREDICTORS = {}

In [10]:
# Build one model file for task369 and package submission.zip.
import shutil

if 'task_ids' not in globals():
    task_ids = [TASK_ID]
assert task_ids == [TASK_ID], task_ids

# Clear stale models from earlier runs before creating this scoped zip.
for old_model_path in OUT_DIR.glob('task*.onnx'):
    old_model_path.unlink()

print('pre-build task ids:', task_ids)
rows = []
for task_id in task_ids:
    task = load_task(DATA_DIR, task_id)
    model_path = OUT_DIR / f'{task_id}.onnx'
    build_info = build_task369_model(task_id, model_path)
    validation_report = validate_task369_model(model_path, task)
    assert validation_report['right'] == validation_report['total'], validation_report
    assert validation_report['file_size_bytes'] < 1_400_000, validation_report['file_size_bytes']
    assert not validation_report['forbidden_ops_present'], validation_report['forbidden_ops_present']
    rows.append({
        'task_id': task_id,
        'saved': True,
        'path': str(model_path),
        'trainer': 'task369_semantic_static_torch_export',
        'model_version': MODEL_VERSION,
        **build_info,
        **validation_report,
    })

zip_path = create_submission_zip(OUT_DIR)
result_df = pd.DataFrame(rows)
display(result_df)
print('selected task369-only tasks:', len(task_ids))
print('models saved:', int(result_df['saved'].sum()) if len(result_df) else 0)
if len(result_df) and 'trainer' in result_df:
    display(result_df['trainer'].fillna('none').value_counts().rename_axis('trainer').reset_index(name='count'))

expected_names = {f'{task_id}.onnx' for task_id in task_ids}
actual_names = {path.name for path in OUT_DIR.glob('task*.onnx')}
print('missing models:', sorted(expected_names - actual_names))
print('extra models:', sorted(actual_names - expected_names))
assert not (expected_names - actual_names), 'missing scoped task models'
assert not (actual_names - expected_names), 'found stale or out-of-scope task models'

# Kaggle looks for /kaggle/working/submission.zip when submitting from a notebook.
submission_zip = Path('/kaggle/working/submission.zip') if Path('/kaggle/working').exists() else Path.cwd() / 'submission.zip'
shutil.copy2(zip_path, submission_zip)
print('family zip:', zip_path)
print('kaggle submission zip:', submission_zip)

pre-build task ids: ['task369']


/tmp/ipykernel_16/481598055.py:252: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,dummy,str(out_path),input_names=['input'],output_names=['output'],opset_version=17,dynamic_axes=None,do_constant_folding=True,dynamo=False)


,task_id,saved,path,trainer,model_version,right,total,accuracy,first_wrong,file_size_bytes,under_1_4mb,forbidden_ops_present,op_counts
0,task369,True,/kaggle/working/working_submission/fill_enclos...,task369_semantic_static_torch_export,fill-additive-nonlocal-multicolor-v11-task369-...,265,265,1.0,None,7259,True,[],"{'Constant': 18, 'ReduceSum': 1, 'Greater': 2,..."


selected task369-only tasks: 1
models saved: 1


,trainer,count
0,task369_semantic_static_torch_export,1


missing models: []
extra models: []
family zip: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_multicolor_task369_v11/submission.zip
kaggle submission zip: /kaggle/working/submission.zip


In [11]:
# Model/version manifest for this notebook run.
run_manifest = {
    'family': FAMILY,
    'subtype': SUBTYPE,
    'model_version': MODEL_VERSION,
    'task_count': len(task_ids),
    'task_ids': task_ids,
    'out_dir': str(OUT_DIR),
    'strategy': 'task369-only semantic static torch export using export_torch_model from task369_solution.ipynb',
    'exportable_predictors': sorted(EXPORTABLE_PREDICTORS.keys()) if 'EXPORTABLE_PREDICTORS' in globals() else [],
    'simulator_only_predictors': sorted(SIMULATOR_ONLY_PREDICTORS.keys()) if 'SIMULATOR_ONLY_PREDICTORS' in globals() else [],
}
run_manifest

{'family': 'fill_enclosed_regions',
 'subtype': 'nonlocal_multicolor',
 'model_version': 'fill-additive-nonlocal-multicolor-v11-task369-semantic-static-rule',
 'task_count': 1,
 'task_ids': ['task369'],
 'out_dir': '/kaggle/working/working_submission/fill_enclosed_regions_nonlocal_multicolor_task369_v11',
 'strategy': 'task369-only semantic static torch export using export_torch_model from task369_solution.ipynb',
 'exportable_predictors': ['task369'],
 'simulator_only_predictors': []}

In [12]:
# Optional: validate saved ONNX models on visible examples.
# This can be slow for large families and requires onnxruntime.
# 'path' in rows: '/kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color_task255_static_onnx_builder_v2_v67/task255.onnx'


validate_rows = []

for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    summary = visible_validation_summary(row['path'], task)
    validate_rows.append({
        'task_id': row['task_id'],
        'right': summary['right'],
        'wrong': summary['wrong'],
    })

print(pd.DataFrame(validate_rows))
print("input and expected output: ", summary['first_wrong_expected'])
print("actual model output: ", summary['first_wrong_actual'])

len of examples:  265
example 10
example 20
example 30
example 40
example 50
example 60
example 70
example 80
example 90
example 100
example 110
example 120
example 130
example 140
example 150
example 160
example 170
example 180
example 190
example 200
example 210
example 220
example 230
example 240
example 250
example 260
   task_id  right  wrong
0  task369    265      0
input and expected output:  None
actual model output:  None


In [13]:
# Architecture, performance, and memory report for saved models.
# This cell expects train_family_task to save one or more ONNX models.
# It reports the metrics the competition cares about: file size, parameter
# count, and memory profile, plus train/test/arc-gen exact-match performance.

report_rows = []
for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    try:
        report = model_report(row['path'], task=task)
        arch = report['architecture']
        mem = report['memory_profile']
        perf = report['performance']
        report_rows.append({
            'task_id': row['task_id'],
            'model_version': MODEL_VERSION,
            'file_size_bytes': arch.get('file_size_bytes'),
            'params': arch.get('params'),
            'nodes': arch.get('nodes'),
            'op_counts': json.dumps(arch.get('op_counts', {}), sort_keys=True),
            'static_memory_bytes': mem.get('static_memory_bytes'),
            'runtime_memory_bytes': mem.get('runtime_memory_bytes'),
            'train_right': perf['train']['right'],
            'train_total': perf['train']['total'],
            'train_accuracy': perf['train']['accuracy'],
            'test_right': perf['test']['right'],
            'test_total': perf['test']['total'],
            'test_accuracy': perf['test']['accuracy'],
            'arc_gen_right': perf['arc_gen']['right'],
            'arc_gen_total': perf['arc_gen']['total'],
            'arc_gen_accuracy': perf['arc_gen']['accuracy'],
            'visible_right': perf['visible_all']['right'],
            'visible_total': perf['visible_all']['total'],
            'visible_accuracy': perf['visible_all']['accuracy'],
        })
    except Exception as exc:
        report_rows.append({
            'task_id': row['task_id'],
            'model_version': MODEL_VERSION,
            'profile_error': repr(exc),
        })

profile_df = pd.DataFrame(report_rows)
display(profile_df)

,task_id,model_version,file_size_bytes,params,nodes,op_counts,static_memory_bytes,runtime_memory_bytes,train_right,train_total,train_accuracy,test_right,test_total,test_accuracy,arc_gen_right,arc_gen_total,arc_gen_accuracy,visible_right,visible_total,visible_accuracy
0,task369,fill-additive-nonlocal-multicolor-v11-task369-...,7259,926,42,"{""Cast"": 4, ""Clip"": 1, ""Concat"": 1, ""Constant""...",59418,75600,3,3,1.0,1,1,1.0,261,261,1.0,265,265,1.0


In [14]:
# Persist run metadata next to the generated models.
if 'profile_df' in globals() and len(profile_df):
    profile_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_profile.csv'
    profile_df.to_csv(profile_path, index=False)
    print('wrote profile:', profile_path)

manifest_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(run_manifest, f, indent=2)
print('wrote manifest:', manifest_path)

wrote profile: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_multicolor_task369_v11/fill_enclosed_regions_fill-additive-nonlocal-multicolor-v11-task369-semantic-static-rule_profile.csv
wrote manifest: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_multicolor_task369_v11/fill_enclosed_regions_fill-additive-nonlocal-multicolor-v11-task369-semantic-static-rule_manifest.json
